# Harness Development

This notebook is the minimal generation loop for the hackathon harness:

1. Load the current CCAT-style question bank.
2. Format prior examples for a requested question type.
3. Route math-like requests through a calculator-enabled generation and verification path.
4. Record generated examples, tool calls, and verifier results in a dataframe for inspection.

In [1]:
import ast
import json
import math
import operator
import os
import re
from getpass import getpass
from typing import Any
from dotenv import load_dotenv

load_dotenv()

import pandas as pd

## Load Current Question Data

The current CSV stores answer choices in separate `choice_a` through `choice_e` columns. Some question types only use A/B/C, so blank D/E values are handled later by the formatter.

In [2]:
DATA_PATH = "ccat_full_question_bank_prompt_stimulus.csv"

QUESTION_COLUMNS = [
    "category",
    "question_type",
    "prompt",
    "stimulus",
    "stimulus_type",
    "choice_a",
    "choice_b",
    "choice_c",
    "choice_d",
    "choice_e",
    "correct_choice_label",
    "explanation",
]

df = pd.read_csv(DATA_PATH)
missing_columns = [column for column in QUESTION_COLUMNS if column not in df.columns]
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

df = df[QUESTION_COLUMNS].copy()

df.head()

,category,question_type,prompt,stimulus,stimulus_type,choice_a,choice_b,choice_c,choice_d,choice_e,correct_choice_label,explanation
0,Verbal,Analogies,NaN,LIBRARY is to BOOKS as ...,text,FOREST is to TREES,MUSEUM is to ARTIFACTS,OVEN is to BAKE,RIVER is to FLOW,TARGET is to AIM,B,B: A library is a place where books are stored...
1,Verbal,Analogies,NaN,OCEAN is to BODY OF WATER as ...,text,MOUNTAIN is to VALLEY,PIANO is to KEYBOARD,NOVEL is to LITERATURE,PARENT is to CHILD,DOG is to WOLF,C,C: An ocean is a specific type or example of a...
2,Verbal,Analogies,NaN,GARAGE is to VEHICLES as ...,text,STORM is to RAIN,ENGINE is to MACHINE,ELEPHANT is to HERD,COOKBOOK is to RECIPES,PREQUEL is to SEQUENCE,D,"D: A cookbook is a collection or ""container"" f..."
3,Verbal,Sentence Completion,"Choose the word or words that, when inserted i...","Despite the team's initial _________, bolstere...",text,optimism,pessimism,skepticism,cynicism,fatalism,A,A: The sentence describes a team that started ...
4,Verbal,Sentence Completion,"Choose the word or words that, when inserted i...",The detective presented the evidence under the...,text,pretense ... elicit,openness ... suppress,veil ... enhance,cover ... conceal,banner ... undermine,A,A: The sentence indicates the detective was pr...


In [3]:
question_type_counts = df["question_type"].value_counts().sort_index()
question_type_counts

question_type
Analogies                                 34
Antonyms                                  16
Applied Quantitative Word Problems        49
Arrangement Logic                          8
Attention to Detail                       24
Basic Numeric Calculation & Comparison    20
Letter-Group Series                        8
Matrix Completion                         20
Number Series                              8
Odd One Out                               32
Percent, Ratio & Proportion               39
Sentence Completion                       68
Syllogisms / Formal Logic                 24
Tables & Graphs                           14
Visual Next-in-Series                     36
Name: count, dtype: int64

## Format Examples For The Model

The generation loop uses only examples from the requested question type. This keeps prompts smaller and makes the style signal cleaner.

In [4]:
CHOICE_COLUMNS = [
    ("A", "choice_a"),
    ("B", "choice_b"),
    ("C", "choice_c"),
    ("D", "choice_d"),
    ("E", "choice_e"),
]

QUESTION_TYPE_ALIASES = {
    # Older labels from Exploration.ipynb / early prompt work.
    "Basic Arithmetic Word Problems": "Applied Quantitative Word Problems",
    "Exact Match Count": "Attention to Detail",
    "Letter Series": "Letter-Group Series",
    "Logic Statements": "Syllogisms / Formal Logic",
    "Logic Statements: True / False / Uncertain": "Syllogisms / Formal Logic",
    "Opposites": "Antonyms",
}

MATH_LIKE_QUESTION_TYPES = {
    "Applied Quantitative Word Problems",
    "Basic Numeric Calculation & Comparison",
    "Number Series",
    "Percent, Ratio & Proportion",
    "Tables & Graphs",
}


def has_value(value) -> bool:
    return pd.notna(value) and str(value).strip() != ""


def resolve_question_type(requested_type: str) -> str:
    """Resolve exact names, known aliases, and unambiguous partial matches."""
    available_types = sorted(df["question_type"].dropna().unique())
    requested_type = requested_type.strip()

    if requested_type in QUESTION_TYPE_ALIASES:
        requested_type = QUESTION_TYPE_ALIASES[requested_type]

    normalized = {question_type.casefold(): question_type for question_type in available_types}
    if requested_type.casefold() in normalized:
        return normalized[requested_type.casefold()]

    partial_matches = [
        question_type
        for question_type in available_types
        if requested_type.casefold() in question_type.casefold()
    ]
    if len(partial_matches) == 1:
        return partial_matches[0]
    if len(partial_matches) > 1:
        raise ValueError(f"Ambiguous question type '{requested_type}'. Matches: {partial_matches}")

    raise ValueError(
        f"Unknown question type '{requested_type}'. Available types: {available_types}"
    )


def is_math_like_question_type(question_type: str) -> bool:
    return resolve_question_type(question_type) in MATH_LIKE_QUESTION_TYPES


def format_text_table(stimulus: str) -> str:
    rows = []
    for raw_row in str(stimulus).split(";"):
        cells = [cell.strip() for cell in raw_row.split("|")]
        if any(cells):
            rows.append(cells)

    if not rows:
        return ""

    column_count = max(len(cells) for cells in rows)
    rows = [cells + [""] * (column_count - len(cells)) for cells in rows]
    widths = [max(len(cells[index]) for cells in rows) for index in range(column_count)]
    separator = "|" + "|".join("-" * (width + 2) for width in widths) + "|"

    table_lines = [separator]
    for cells in rows:
        table_lines.append(
            "| "
            + " | ".join(f"{cells[index]:<{widths[index]}}" for index in range(column_count))
            + " |"
        )
        table_lines.append(separator)
    return "\n".join(table_lines)


def format_stimulus(row: pd.Series) -> str:
    stimulus = row.get("stimulus", "")
    if not has_value(stimulus):
        return ""

    stimulus_type = row.get("stimulus_type", "")
    if stimulus_type == "text_table":
        return format_text_table(stimulus)
    if stimulus_type == "image":
        return f"IMAGE STIMULUS: {str(stimulus).strip()}"
    return str(stimulus).strip()


def format_choices(row: pd.Series) -> str:
    choices = []
    for letter, column in CHOICE_COLUMNS:
        value = row.get(column, "")
        if has_value(value):
            choices.append(f"{letter}: {str(value).strip()}")
    return "\n".join(choices)


def format_question_for_generation(row: pd.Series) -> str:
    parts = [f"QUESTION TYPE: {row.get('question_type', '')}"]

    prompt = row.get("prompt", "")
    if has_value(prompt):
        parts.extend(["", str(prompt).strip()])

    stimulus = format_stimulus(row)
    if stimulus:
        parts.extend(["", stimulus])

    choices = format_choices(row)
    if choices:
        parts.extend(["", choices])

    parts.extend([
        "",
        "CORRECT CHOICE:",
        str(row.get("correct_choice_label", "")).strip(),
        "",
        "EXPLANATION:",
        str(row.get("explanation", "")).strip(),
    ])
    return "\n".join(parts)


def format_questions_for_generation(question_df: pd.DataFrame) -> str:
    formatted_examples = [
        format_question_for_generation(row)
        for _, row in question_df.iterrows()
    ]
    separator = "\n\n" + "=" * 100 + "\n\n"
    return separator.join(formatted_examples)


def build_labeled_question_corpus(
    requested_type: str,
    examples_per_type: int = 8,
    random_state: int = 7,
) -> str:
    resolved_type = resolve_question_type(requested_type)
    examples = df.loc[df["question_type"] == resolved_type].copy()

    if examples.empty:
        raise ValueError(f"No examples found for question type: {resolved_type}")

    if len(examples) > examples_per_type:
        examples = examples.sample(n=examples_per_type, random_state=random_state)

    return format_questions_for_generation(examples)

In [5]:
# Non-LLM smoke test: preview the examples that will be sent for one requested type.
preview_type = "Analogies"
preview_corpus = build_labeled_question_corpus(preview_type, examples_per_type=3)
print(preview_corpus[:3000])

QUESTION TYPE: Analogies

GARAGE is to VEHICLES as ...

A: STORM is to RAIN
B: ENGINE is to MACHINE
C: ELEPHANT is to HERD
D: COOKBOOK is to RECIPES
E: PREQUEL is to SEQUENCE

CORRECT CHOICE:
D

EXPLANATION:
D: A cookbook is a collection or "container" for recipes, similar to a garage for vehicles. The analogy is about something that holds or organizes specific items. Options A (cause-effect), B (part-whole), C (member-group), E (part-sequence) are distractors.


QUESTION TYPE: Analogies

SAFE is to SECURE as PROTECT is to _____.

A: lock
B: guard
C: conserve
D: run
E: campaign

CORRECT CHOICE:
B

EXPLANATION:
B: Option (B) is the best choice. Safe and secure have similar meanings. We need to find the option that is most similar to protect. Guard is the best choice.


QUESTION TYPE: Analogies

OCEAN is to BODY OF WATER as ...

A: MOUNTAIN is to VALLEY
B: PIANO is to KEYBOARD
C: NOVEL is to LITERATURE
D: PARENT is to CHILD
E: DOG is to WOLF

CORRECT CHOICE:
C

EXPLANATION:
C: An ocean i

## LangChain Generation Chain

Set `OPENAI_API_KEY` in your environment before running the notebook, or this cell will ask for it without displaying it. You can override the model with `OPENAI_MODEL`; otherwise it uses `gpt-4o-mini`, matching the earlier exploration work.

In [6]:
from langchain_core.messages import ToolMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
TEMPERATURE = float(os.getenv("OPENAI_TEMPERATURE", "0.2"))

if not os.getenv("OPENAI_API_KEY"):
    entered_key = getpass("OpenAI API key: ")
    if not entered_key:
        raise RuntimeError("Set OPENAI_API_KEY or enter a key to run model generation.")
    os.environ["OPENAI_API_KEY"] = entered_key

print(f"Using model: {MODEL_NAME}")

Using model: gpt-4o-mini


## Calculator Tool

The calculator intentionally evaluates only arithmetic expressions. The tool catches unsafe/invalid expressions and reports the error back to the model instead of executing arbitrary code.

In [7]:
SAFE_BINARY_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Pow: operator.pow,
}
SAFE_UNARY_OPERATORS = {
    ast.UAdd: operator.pos,
    ast.USub: operator.neg,
}
SAFE_FUNCTIONS = {
    "abs": abs,
    "ceil": math.ceil,
    "floor": math.floor,
    "round": round,
    "sqrt": math.sqrt,
}
SAFE_NAMES = {
    "e": math.e,
    "pi": math.pi,
}


def normalize_calculator_expression(expression: str) -> str:
    expression = str(expression).strip()
    expression = expression.replace("$", "")
    expression = re.sub(r"(?<=\d),(?=\d{3}(\D|$))", "", expression)
    expression = re.sub(
        r"(?<![\w.])(\d+(?:\.\d+)?)\s*%(?!\s*\d)",
        r"(\1 / 100)",
        expression,
    )
    return expression


def evaluate_safe_ast(node: ast.AST) -> float:
    if isinstance(node, ast.Expression):
        return evaluate_safe_ast(node.body)

    if isinstance(node, ast.Constant):
        if isinstance(node.value, bool) or not isinstance(node.value, (int, float)):
            raise ValueError("Only numeric constants are allowed")
        return node.value

    if isinstance(node, ast.Name):
        if node.id not in SAFE_NAMES:
            raise ValueError(f"Unknown name: {node.id}")
        return SAFE_NAMES[node.id]

    if isinstance(node, ast.BinOp):
        operator_type = type(node.op)
        if operator_type not in SAFE_BINARY_OPERATORS:
            raise ValueError(f"Unsupported operator: {operator_type.__name__}")
        left = evaluate_safe_ast(node.left)
        right = evaluate_safe_ast(node.right)
        if operator_type is ast.Pow and abs(right) > 10:
            raise ValueError("Exponents greater than 10 are not allowed")
        return SAFE_BINARY_OPERATORS[operator_type](left, right)

    if isinstance(node, ast.UnaryOp):
        operator_type = type(node.op)
        if operator_type not in SAFE_UNARY_OPERATORS:
            raise ValueError(f"Unsupported unary operator: {operator_type.__name__}")
        return SAFE_UNARY_OPERATORS[operator_type](evaluate_safe_ast(node.operand))

    if isinstance(node, ast.Call):
        if not isinstance(node.func, ast.Name) or node.func.id not in SAFE_FUNCTIONS:
            raise ValueError("Only allowlisted math functions can be called")
        if node.keywords:
            raise ValueError("Keyword arguments are not supported")
        if len(node.args) > 2:
            raise ValueError("Calculator functions accept at most two arguments")
        args = [evaluate_safe_ast(arg) for arg in node.args]
        return SAFE_FUNCTIONS[node.func.id](*args)

    raise ValueError(f"Unsupported expression element: {type(node).__name__}")


def format_calculator_result(result: float) -> str:
    if isinstance(result, float) and not math.isfinite(result):
        raise ValueError("Result is not finite")
    if abs(result - round(result)) < 1e-12:
        return str(int(round(result)))
    return format(result, ".12g")


def safe_calculate_expression(expression: str) -> str:
    normalized_expression = normalize_calculator_expression(expression)
    if not normalized_expression:
        raise ValueError("Expression is empty")
    if len(normalized_expression) > 240:
        raise ValueError("Expression is too long")

    parsed_expression = ast.parse(normalized_expression, mode="eval")
    result = evaluate_safe_ast(parsed_expression)
    return format_calculator_result(result)


@tool
def calculate(expression: str) -> str:
    """Evaluate a safe arithmetic expression. Use this for numeric checks; write percentages as 46% or 46 / 100."""
    try:
        return safe_calculate_expression(expression)
    except Exception as exc:
        return f"ERROR: {exc}"

In [8]:
# Non-LLM guardrail smoke tests for the calculator tool.
assert safe_calculate_expression("4.50 * 4") == "18"
assert safe_calculate_expression("57,000 * 46%") == "26220"
assert safe_calculate_expression("round(10 / 3, 2)") == "3.33"
assert is_math_like_question_type("Applied Quantitative Word Problems")
assert not is_math_like_question_type("Analogies")

try:
    safe_calculate_expression("__import__('os').system('echo unsafe')")
    raise AssertionError("Unsafe expression was not rejected")
except ValueError:
    pass

print("calculator guardrail smoke tests passed")

calculator guardrail smoke tests passed


## Prompts And Tool Loop

Non-math requests still use the simple text generation prompt. Math-like requests use a structured JSON prompt plus the calculator tool during generation and verification.

In [9]:
GENERATION_PROMPT_TEMPLATE = """
You are a CCAT-style question-generation assistant.

Your task is to generate one new multiple-choice question that belongs to the requested question type.

You will be given:
1. A corpus of existing example questions from the requested question type.
2. The requested question type name.

Use the corpus to learn the style, difficulty, structure, answer-choice format, and reasoning pattern for the requested question type.

Do not copy, lightly paraphrase, or reuse any existing question from the corpus. Generate a genuinely new question that could plausibly belong in the same dataset.

Requested question type:
{QUESTION_TYPE}

Corpus of labeled example questions:
{LABELED_QUESTION_CORPUS}

Question-type rules:
- If the corpus examples use exactly three choices, generate exactly three choices labeled A, B, and C.
- Otherwise, generate exactly five choices labeled A, B, C, D, and E.
- If the examples include a text comparison table, include a similar text table in the generated question.
- If the examples are image-stimulus questions, do not invent an image filename. Generate a text-only version that preserves the reasoning pattern unless image generation is explicitly requested later.

General requirements:
- Generate exactly one question.
- The question must clearly fit the requested question type.
- Include exactly one correct answer.
- Make all incorrect choices plausible but clearly wrong.
- Match the style and approximate difficulty of the corpus examples for the requested question type.
- Avoid ambiguity.
- Avoid requiring outside knowledge.
- Do not mention the corpus.
- Do not explain that the question is new.
- Do not include any metadata unless requested in the output format.

Return your answer in exactly this format:

QUESTION:
[question text]

CHOICES:
A: [choice A]
B: [choice B]
C: [choice C]
[include D and E only when the question uses five choices]

ANSWER:
[correct choice letter]

EXPLANATION:
[brief explanation of why the correct choice is correct and why the other choices are not]
"""

MATH_GENERATION_PROMPT_TEMPLATE = """
You are a CCAT-style math question-generation assistant with access to a calculator tool named calculate.

Your task is to generate one new multiple-choice question that belongs to the requested math-like question type.

Requested question type:
{QUESTION_TYPE}

Corpus of labeled example questions:
{LABELED_QUESTION_CORPUS}

Use the corpus to learn the style, difficulty, structure, answer-choice format, and reasoning pattern for the requested question type.

Calculator requirements:
- Use the calculate tool for every non-trivial arithmetic step before choosing the correct answer.
- Do not guess arithmetic mentally when a calculation is needed.
- Put the expressions you checked in calculation_notes.
- Return valid JSON. Every displayed answer choice value must be a JSON string in double quotes, especially numbers with commas, currency, or units such as 18,000, $18.00, or 46%.

Question requirements:
- Generate exactly one question.
- Include exactly one correct answer.
- Make all incorrect choices plausible but clearly wrong.
- Match the style and approximate difficulty of the corpus examples.
- Avoid ambiguity and outside knowledge.
- Do not mention the corpus or the calculator.

Return JSON only, with this exact shape:
{{
  "question": "[question text]",
  "choices": {{
    "A": "[choice A]",
    "B": "[choice B]",
    "C": "[choice C]",
    "D": "[choice D]",
    "E": "[choice E]"
  }},
  "answer": "[correct choice letter]",
  "explanation": "[brief explanation of why the correct choice is correct and why the other choices are not]",
  "calculation_notes": ["[calculator expression checked -> result]"]
}}
"""

MATH_VERIFICATION_PROMPT_TEMPLATE = """
You are a strict verifier for generated CCAT-style math questions. You have access to a calculator tool named calculate.

Requested question type:
{QUESTION_TYPE}

Generated question JSON:
{GENERATED_OUTPUT_JSON}

Verification task:
- Check whether the listed correct answer follows from the numbers in the generated question.
- Use the calculate tool for arithmetic checks.
- If the question cannot be fully checked from the provided text, return warning rather than pass.
- If the answer is numerically wrong, return fail.

Return JSON only, with this exact shape:
{{
  "verdict": "pass | warning | fail",
  "checked_expression": "[main calculator expression checked, or blank]",
  "expected_answer": "[numeric/text answer you verified, or blank]",
  "model_answer": "[answer choice and value from the generated question, or blank]",
  "notes": "[short explanation of the verdict]"
}}
"""

MATH_JSON_REPAIR_PROMPT_TEMPLATE = """
You previously returned invalid JSON for a generated math question.

Requested question type:
{QUESTION_TYPE}

JSON parser error:
{PARSE_ERROR}

Invalid response:
{INVALID_RESPONSE}

Fix the response so it is valid JSON only.

Rules:
- Do not wrap the JSON in markdown fences.
- Do not add commentary before or after the JSON.
- Preserve the generated question, choices, answer, explanation, and calculation_notes unless a minimal syntax fix is required.
- Every displayed answer choice value must be a JSON string in double quotes.
- Numbers with commas, currency symbols, percentages, or units must be strings, not bare JSON numbers.
- The output must parse with json.loads.
"""

gen_prompt = ChatPromptTemplate.from_template(GENERATION_PROMPT_TEMPLATE)
math_gen_prompt = ChatPromptTemplate.from_template(MATH_GENERATION_PROMPT_TEMPLATE)
math_verification_prompt = ChatPromptTemplate.from_template(MATH_VERIFICATION_PROMPT_TEMPLATE)
math_json_repair_prompt = ChatPromptTemplate.from_template(MATH_JSON_REPAIR_PROMPT_TEMPLATE)

gen_llm = ChatOpenAI(model=MODEL_NAME, temperature=TEMPERATURE)
output_parser = StrOutputParser()
gen_chain = gen_prompt | gen_llm | output_parser
CALCULATOR_TOOLS = [calculate]

In [10]:
def message_content_to_text(content: Any) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        text_parts = []
        for item in content:
            if isinstance(item, dict) and "text" in item:
                text_parts.append(str(item["text"]))
            else:
                text_parts.append(str(item))
        return "\n".join(text_parts)
    return str(content)


def extract_json_candidate(content: Any) -> tuple[str | None, str]:
    text = message_content_to_text(content).strip()
    fenced_match = re.search(r"```(?:json)?\s*(.*?)```", text, flags=re.IGNORECASE | re.DOTALL)
    if fenced_match:
        return fenced_match.group(1).strip(), text

    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None, text
    return text[start : end + 1], text


def parse_json_response_with_error(content: Any) -> tuple[dict[str, Any] | None, str, str]:
    json_candidate, raw_text = extract_json_candidate(content)
    if json_candidate is None:
        return None, "No JSON object found in model response.", raw_text

    try:
        parsed = json.loads(json_candidate)
    except json.JSONDecodeError as exc:
        return None, f"{exc.msg} at line {exc.lineno}, column {exc.colno}", raw_text

    if not isinstance(parsed, dict):
        return None, "JSON parsed successfully but was not an object.", raw_text

    return parsed, "", raw_text


def parse_json_response(content: Any) -> dict[str, Any] | None:
    parsed, _, _ = parse_json_response_with_error(content)
    return parsed


def render_generated_question(parsed_output: dict[str, Any] | None, fallback_text: str = "") -> str:
    if not parsed_output:
        return fallback_text

    question = str(parsed_output.get("question", "")).strip()
    choices = parsed_output.get("choices", {})
    answer = str(parsed_output.get("answer", "")).strip().upper()
    explanation = str(parsed_output.get("explanation", "")).strip()

    if isinstance(choices, list):
        choices = {chr(65 + index): value for index, value in enumerate(choices)}
    if not isinstance(choices, dict):
        choices = {}

    choice_lines = []
    for letter in ["A", "B", "C", "D", "E"]:
        if has_value(choices.get(letter)):
            choice_lines.append(f"{letter}: {str(choices[letter]).strip()}")

    return "\n\n".join([
        "QUESTION:\n" + question,
        "CHOICES:\n" + "\n".join(choice_lines),
        "ANSWER:\n" + answer,
        "EXPLANATION:\n" + explanation,
    ])


def invoke_prompt_with_tools(
    prompt: ChatPromptTemplate,
    variables: dict[str, Any],
    tools,
    max_tool_rounds: int = 6,
) -> tuple[str, list[dict[str, Any]]]:
    messages = prompt.invoke(variables).to_messages()
    llm_with_tools = gen_llm.bind_tools(tools)
    tool_registry = {tool_obj.name: tool_obj for tool_obj in tools}
    tool_records = []

    for _ in range(max_tool_rounds):
        ai_message = llm_with_tools.invoke(messages)
        messages.append(ai_message)
        tool_calls = getattr(ai_message, "tool_calls", None) or []

        if not tool_calls:
            return message_content_to_text(ai_message.content), tool_records

        for tool_call in tool_calls:
            tool_name = tool_call.get("name")
            tool_args = tool_call.get("args", {})
            tool_call_id = tool_call.get("id")
            tool_obj = tool_registry.get(tool_name)

            if tool_obj is None:
                tool_result = f"ERROR: Unknown tool {tool_name}"
            else:
                tool_result = tool_obj.invoke(tool_args)

            tool_records.append({
                "tool_name": tool_name,
                "args": tool_args,
                "result": str(tool_result),
            })
            messages.append(ToolMessage(content=str(tool_result), tool_call_id=tool_call_id))

    raise RuntimeError(f"Tool loop exceeded {max_tool_rounds} rounds")


def preview_generation_prompt(requested_type: str, examples_per_type: int = 6) -> str:
    resolved_type = resolve_question_type(requested_type)
    corpus = build_labeled_question_corpus(resolved_type, examples_per_type=examples_per_type)
    prompt = math_gen_prompt if is_math_like_question_type(resolved_type) else gen_prompt
    return prompt.format(
        QUESTION_TYPE=resolved_type,
        LABELED_QUESTION_CORPUS=corpus,
    )


def normalize_verification(verification: dict[str, Any] | None) -> dict[str, Any]:
    if not isinstance(verification, dict):
        verification = {}

    verdict = str(verification.get("verdict", "warning")).strip().lower()
    if verdict not in {"pass", "warning", "fail", "not_applicable"}:
        notes = str(verification.get("notes", "")).strip()
        verification["notes"] = (
            f"Verifier returned an unrecognized verdict ({verdict!r}). "
            + notes
        ).strip()
        verdict = "warning"

    return {
        "verdict": verdict,
        "checked_expression": str(verification.get("checked_expression", "")).strip(),
        "expected_answer": str(verification.get("expected_answer", "")).strip(),
        "model_answer": str(verification.get("model_answer", "")).strip(),
        "notes": str(verification.get("notes", "")).strip(),
    }


def repair_math_json_response(
    resolved_type: str,
    invalid_response: str,
    parse_error: str,
    max_repair_attempts: int = 2,
) -> tuple[dict[str, Any] | None, str, list[dict[str, Any]]]:
    repair_records = []
    current_response = invalid_response
    current_error = parse_error

    for attempt_number in range(1, max_repair_attempts + 1):
        repair_messages = math_json_repair_prompt.invoke({
            "QUESTION_TYPE": resolved_type,
            "PARSE_ERROR": current_error,
            "INVALID_RESPONSE": current_response,
        }).to_messages()
        repair_message = gen_llm.invoke(repair_messages)
        repaired_text = message_content_to_text(repair_message.content)
        parsed_output, next_error, raw_repaired_text = parse_json_response_with_error(repaired_text)

        repair_records.append({
            "attempt": attempt_number,
            "input_parse_error": current_error,
            "raw_repair_response": raw_repaired_text,
            "success": parsed_output is not None,
            "output_parse_error": next_error,
        })

        if parsed_output is not None:
            return parsed_output, raw_repaired_text, repair_records

        current_response = raw_repaired_text
        current_error = next_error

    return None, current_response, repair_records


def verify_math_question(
    resolved_type: str,
    parsed_output: dict[str, Any],
) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    verification_text, verification_tool_calls = invoke_prompt_with_tools(
        math_verification_prompt,
        {
            "QUESTION_TYPE": resolved_type,
            "GENERATED_OUTPUT_JSON": json.dumps(parsed_output, ensure_ascii=False, indent=2),
        },
        CALCULATOR_TOOLS,
    )
    verification = parse_json_response(verification_text)
    if verification is None:
        verification = {
            "verdict": "warning",
            "checked_expression": "",
            "expected_answer": "",
            "model_answer": "",
            "notes": f"Verifier did not return parseable JSON: {verification_text}",
        }
    return normalize_verification(verification), verification_tool_calls


def generate_standard_question(
    resolved_type: str,
    corpus: str,
) -> dict[str, Any]:
    generated_question = gen_chain.invoke({
        "QUESTION_TYPE": resolved_type,
        "LABELED_QUESTION_CORPUS": corpus,
    })
    return {
        "used_tool_path": False,
        "generated_question": generated_question,
        "parsed_output": None,
        "tool_calls": [],
        "verification": normalize_verification({
            "verdict": "not_applicable",
            "checked_expression": "",
            "expected_answer": "",
            "model_answer": "",
            "notes": "Non-math question type skipped calculator verification.",
        }),
    }


def generate_math_question(
    resolved_type: str,
    corpus: str,
) -> dict[str, Any]:
    json_repair_attempts = []
    generation_text, generation_tool_calls = invoke_prompt_with_tools(
        math_gen_prompt,
        {
            "QUESTION_TYPE": resolved_type,
            "LABELED_QUESTION_CORPUS": corpus,
        },
        CALCULATOR_TOOLS,
    )
    parsed_output, parse_error, raw_generation_text = parse_json_response_with_error(generation_text)
    final_generation_text = raw_generation_text

    if parsed_output is None:
        parsed_output, final_generation_text, json_repair_attempts = repair_math_json_response(
            resolved_type,
            raw_generation_text,
            parse_error,
        )

    generated_question = render_generated_question(parsed_output, fallback_text=final_generation_text)

    if parsed_output is None:
        verification = normalize_verification({
            "verdict": "warning",
            "checked_expression": "",
            "expected_answer": "",
            "model_answer": "",
            "notes": "Generated math response was not parseable JSON after repair attempts, so calculator verification was skipped.",
        })
        verification_tool_calls = []
    else:
        verification, verification_tool_calls = verify_math_question(resolved_type, parsed_output)

    return {
        "used_tool_path": True,
        "generated_question": generated_question,
        "parsed_output": parsed_output,
        "raw_generation_response": raw_generation_text,
        "final_generation_response": final_generation_text,
        "json_repair_attempts": json_repair_attempts,
        "tool_calls": generation_tool_calls + verification_tool_calls,
        "verification": verification,
    }


def run_generation_request(requested_type: str, examples_per_type: int = 6) -> dict[str, Any]:
    resolved_type = resolve_question_type(requested_type)
    corpus = build_labeled_question_corpus(resolved_type, examples_per_type=examples_per_type)

    if is_math_like_question_type(resolved_type):
        result = generate_math_question(resolved_type, corpus)
    else:
        result = generate_standard_question(resolved_type, corpus)

    verification = result.get("verification", {})
    return {
        "requested_type": requested_type,
        "resolved_type": resolved_type,
        "used_tool_path": result.get("used_tool_path", False),
        "generated_question": result.get("generated_question", ""),
        "parsed_output": result.get("parsed_output"),
        "raw_generation_response": result.get("raw_generation_response", result.get("generated_question", "")),
        "final_generation_response": result.get("final_generation_response", result.get("generated_question", "")),
        "json_repair_attempts": result.get("json_repair_attempts", []),
        "tool_calls": result.get("tool_calls", []),
        "verification_verdict": verification.get("verdict", ""),
        "verification_notes": verification.get("notes", ""),
        "verification": verification,
    }


def generate_question(requested_type: str, examples_per_type: int = 6) -> str:
    return run_generation_request(requested_type, examples_per_type=examples_per_type)["generated_question"]


def generate_questions_for_types(requested_types, examples_per_type: int = 6) -> pd.DataFrame:
    results = [
        run_generation_request(requested_type, examples_per_type=examples_per_type)
        for requested_type in requested_types
    ]
    return pd.DataFrame(results)

In [11]:
# Optional debug cell: confirm LangChain is substituting values into the prompt.
print(preview_generation_prompt("Applied Quantitative Word Problems", examples_per_type=2)[:2500])

Human: 
You are a CCAT-style math question-generation assistant with access to a calculator tool named calculate.

Your task is to generate one new multiple-choice question that belongs to the requested math-like question type.

Requested question type:
Applied Quantitative Word Problems

Corpus of labeled example questions:
QUESTION TYPE: Applied Quantitative Word Problems

A cell can replicate itself 3,800 times every 30 seconds. At this rate, it would take ______ seconds to replicate itself 13,500 times.

A: 106.6
B: 108.6
C: 109.8
D: 111.8
E: 112

CORRECT CHOICE:
A

EXPLANATION:
A: We can set up a proportion to solve this question. 3800/30 = 13500/x. We cross-multiply to get 3800x = 405,000. We divide both sides by 3800 to get 106.6 seconds.


QUESTION TYPE: Applied Quantitative Word Problems

A bookstore sells 80 novels per day for 20 days in a month. It also sells 40 textbooks per day for the remaining 10 days of the month. What is the total number of books sold by the bookstore 

## Run The Main Generation Loop

This is the current harness loop: each requested type is resolved to the current dataset's `question_type`, prior examples are formatted, and the model generates one new question. Math-like requests use the calculator-enabled path and include verifier results in `generated_examples`.

In [12]:
EXAMPLE_REQUESTS = [
    "Analogies",
    "Applied Quantitative Word Problems",
    "Percent, Ratio & Proportion",
    "Basic Numeric Calculation & Comparison",
    "Syllogisms / Formal Logic",
]

generated_examples = generate_questions_for_types(EXAMPLE_REQUESTS, examples_per_type=6)

for _, row in generated_examples.iterrows():
    print("\n" + "=" * 100)
    print(f"REQUESTED: {row['requested_type']} -> {row['resolved_type']}")
    print(f"TOOL PATH: {row['used_tool_path']} | VERDICT: {row['verification_verdict']}")
    if row["json_repair_attempts"]:
        print(f"JSON REPAIR ATTEMPTS: {len(row['json_repair_attempts'])}")
    if row["verification_notes"]:
        print(f"VERIFIER NOTES: {row['verification_notes']}")
    print("=" * 100)
    print(row["generated_question"])

generated_examples[[
    "requested_type",
    "resolved_type",
    "used_tool_path",
    "json_repair_attempts",
    "verification_verdict",
    "verification_notes",
    "tool_calls",
]]


REQUESTED: Analogies -> Analogies
TOOL PATH: False | VERDICT: not_applicable
VERIFIER NOTES: Non-math question type skipped calculator verification.
QUESTION:
BIRD is to FLY as FISH is to _____.

CHOICES:
A: swim  
B: crawl  
C: walk  
D: dive  
E: float  

ANSWER:
A

EXPLANATION:
A: The relationship here is about the primary mode of movement for each animal. A bird's primary mode of movement is to fly, just as a fish's primary mode of movement is to swim. The other options do not represent the correct movement associated with fish: B (crawl) and C (walk) are not applicable to fish, D (dive) is a specific action rather than the general movement, and E (float) describes a state rather than a mode of movement.

REQUESTED: Applied Quantitative Word Problems -> Applied Quantitative Word Problems
TOOL PATH: True | VERDICT: pass
VERIFIER NOTES: The calculations confirm that Emily spent a total of $29.75, which matches the provided answer.
QUESTION:
Emily bought 4 shirts, each costing $5.75,

,requested_type,resolved_type,used_tool_path,json_repair_attempts,verification_verdict,verification_notes,tool_calls
0,Analogies,Analogies,False,[],not_applicable,Non-math question type skipped calculator veri...,[]
1,Applied Quantitative Word Problems,Applied Quantitative Word Problems,True,[],pass,The calculations confirm that Emily spent a to...,"[{'tool_name': 'calculate', 'args': {'expressi..."
2,"Percent, Ratio & Proportion","Percent, Ratio & Proportion",True,[],fail,The calculated percentage of finishers from Te...,"[{'tool_name': 'calculate', 'args': {'expressi..."
3,Basic Numeric Calculation & Comparison,Basic Numeric Calculation & Comparison,True,[],pass,The calculations confirm that the truck delive...,"[{'tool_name': 'calculate', 'args': {'expressi..."
4,Syllogisms / Formal Logic,Syllogisms / Formal Logic,False,[],not_applicable,Non-math question type skipped calculator veri...,[]
